In [1]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5"

In [2]:
# Helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

In [ ]:
import json


def generate_dataset():
    prompt = """
You are an AWS expert. Generate an evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
        "format": "json" or "python" or "regex",
        "solution_criteria":"Must include runtime, memory size, timeout, and basic structure for the required configuration"
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""
    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])
    return json.loads(text)

In [17]:
dataset = generate_dataset()

dataset

[{'task': "Parse an AWS S3 object key to extract the bucket name, folder path, and file name from a full S3 URI like 's3://my-bucket/folder/subfolder/file.txt'",
  'format': 'regex'},
 {'task': 'Create a CloudFormation template JSON object that defines an AWS Lambda function with basic configuration including function name, runtime, and IAM role ARN',
  'format': 'json'},
 {'task': 'Write a Python function that takes an AWS CloudWatch log group name and returns a formatted timestamp in ISO 8601 format for querying logs from the last 24 hours',
  'format': 'python'}]

In [18]:
with open("dataset.json", "w") as f:
    json.dump(dataset, f, indent=2)

In [26]:
def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result."""
    prompt= f"""
    Please solve the following task:
    
    {test_case["task"]}
    
    * Respond only with Python, JSON or plain regex
    * Do not add any comments, commentary or explanation
    """
    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```code")
    output = chat(messages, stop_sequences=["```"])
    return output

In [22]:
# Function to grade a test case + output using a model
def grade_by_model(test_case, output):
    eval_prompt = f"""
You are an expert AWS code reviewer. Your task is to evaluate the following AI-generated solution.

Original Task:
<task>
{test_case["task"]}
</task>

Solution to Evaluate:
<solution>
{output}
</solution>

Output Format
Provide your evaluation as a structured JSON object with the following fields, in this specific order:
- "strengths": An array of 1-3 key strengths
- "weaknesses": An array of 1-3 key areas for improvement
- "reasoning": A concise explanation of your overall assessment
- "score": A number between 1-10

Respond with JSON. Keep your response concise and direct.
Example response shape:
{{
    "strengths": string[],
    "weaknesses": string[],
    "reasoning": string,
    "score": number
}}
    """

    messages = []
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```code")
    eval_text = chat(messages, stop_sequences=["```"])
    return json.loads(eval_text)

In [15]:
# Functions to validate the output structure
import re
import ast


def validate_json(text):
    try:
        json.loads(text.strip())
        return 10
    except json.JSONDecodeError:
        return 0


def validate_python(text):
    try:
        ast.parse(text.strip())
        return 10
    except SyntaxError:
        return 0


def validate_regex(text):
    try:
        re.compile(text.strip())
        return 10
    except re.error:
        return 0


def grade_syntax(response, test_case):
    format = test_case["format"]
    if format == "json":
        return validate_json(response)
    elif format == "python":
        return validate_python(response)
    else:
        return validate_regex(response)

In [23]:
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)

    # TODO - Implement grading, hard coded for now
    model_grade = grade_by_model(test_case, output)
    model_score = model_grade["score"]
    reasoning = model_grade["reasoning"]
    
    syntax_score = grade_syntax(output, test_case)
    
    score = (model_score + syntax_score)/2

    return {
        "output": output,
        "test_case": test_case,
        "score": score,
        "reasoning": reasoning,
    }

In [24]:
from statistics import mean

def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []
    
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)
    
    average_score = mean([result["score"] for result in results])
    print(f"Average Score: {average_score}")
        
    return results

In [27]:
with open("dataset.json") as f:
    dataset = json.load(f)

results = run_eval(dataset)

Average Score: 6.5


In [12]:
print(json.dumps(results, indent=2))

[
  {
    "output": "# AWS S3 Bucket Name Parser\n\nHere's a Python function that parses an S3 bucket name from an S3 URI:\n\n```python\ndef parse_s3_bucket_name(s3_uri: str) -> str:\n    \"\"\"\n    Parse an AWS S3 bucket name from an S3 URI.\n    \n    Args:\n        s3_uri (str): An S3 URI in the format 's3://bucket-name/key/path'\n    \n    Returns:\n        str: The bucket name\n    \n    Raises:\n        ValueError: If the URI is not a valid S3 URI\n    \n    Examples:\n        >>> parse_s3_bucket_name('s3://my-bucket/path/to/file.txt')\n        'my-bucket'\n        >>> parse_s3_bucket_name('s3://another-bucket/folder/')\n        'another-bucket'\n    \"\"\"\n    # Remove leading/trailing whitespace\n    s3_uri = s3_uri.strip()\n    \n    # Validate S3 URI format\n    if not s3_uri.startswith('s3://'):\n        raise ValueError(f\"Invalid S3 URI format: {s3_uri}. Expected format: 's3://bucket-name/key/path'\")\n    \n    # Remove 's3://' prefix\n    uri_without_prefix = s3_uri[5: